In [ ]:
import psycopg2
import pandas as pd
import networkx as nx

In [3]:
# Connect to PostgreSQL and fetch data
conn = psycopg2.connect("dbname=sui_indexer user=postgres password=56904628")
cur = conn.cursor()

Count transactions

In [28]:
cur.execute("""
SELECT
    (SELECT COUNT(*) FROM transactions) AS total_tx,
    (SELECT SUM(user_tx_count) FROM checkpoints) AS user_tx,
    (SELECT SUM(system_tx_count) FROM checkpoints) AS system_tx;
""")

total_tx, user_tx, system_tx = cur.fetchone()

print("Total transactions:", total_tx)
print("Total user tx:", user_tx)
print("Total system tx:", system_tx)


Total transactions: 779434
Total user tx: 8
Total system tx: 779426


Write set

In [18]:
cur.execute("""SELECT
        t.transaction_digest,
        oc.address,
        c.timestamp
    FROM transactions t
    JOIN checkpoints c
        ON t.checkpoint_sequence = c.sequence_number
    LEFT JOIN object_changes oc
        ON t.transaction_digest = oc.transaction_digest
    WHERE
        oc.change_type IN ('mutated', 'created', 'deleted')
        OR
        t.inputs_shared_mut > 0
        OR t.inputs_receiving > 0
        OR t.inputs_imm_or_owned > 0
    ORDER BY c.timestamp DESC;""")
rows = cur.fetchall()

write_transactions = []
write_object_addresses = []
write_timestamps = []

for tx_digest, address, timestamp in rows:
    write_transactions.append(tx_digest)
    write_object_addresses.append(address)
    write_timestamps.append(timestamp)


In [20]:
len(write_transactions)

6

Read set

In [21]:
cur.execute("""SELECT
    t.transaction_digest,
    c.timestamp,
    oc.address
FROM transactions t
JOIN checkpoints c
    ON t.checkpoint_sequence = c.sequence_number
LEFT JOIN object_changes oc
    ON t.transaction_digest = oc.transaction_digest
WHERE
    -- First: transaction does NOT write anything
    t.inputs_shared_mut = 0
    AND t.inputs_receiving = 0
    AND t.inputs_funds_withdrawal = 0
    AND (
        oc.change_type IS NULL             -- no object modifications
        OR oc.change_type NOT IN ('mutated', 'created', 'deleted')
    )
    AND
    (
        -- These make it a READ:
        t.inputs_pure > 0
        OR t.inputs_shared_ro > 0
        OR t.inputs_imm_or_owned > 0
    )
ORDER BY c.timestamp DESC;""")
rows = cur.fetchall()

read_transactions = []
read_object_addresses = []
read_timestamps = []

for tx_digest, address, timestamp in rows:
    read_transactions.append(tx_digest)
    read_object_addresses.append(address)
    read_timestamps.append(timestamp)


In [23]:
len(read_transactions)

3

Conflict graphs

Real conflicts

In [ ]:
cur.execute("""SELECT
    oc1.transaction_digest AS tx1,
    oc2.transaction_digest AS tx2,
    oc1.address,
    oc1.input_version AS version,
    c.sequence_number AS checkpoint
FROM object_changes oc1
JOIN object_changes oc2
    ON oc1.address = oc2.address
    AND oc1.transaction_digest < oc2.transaction_digest
    AND oc1.input_version = oc2.input_version
JOIN transactions t1
    ON oc1.transaction_digest = t1.transaction_digest
JOIN transactions t2
    ON oc2.transaction_digest = t2.transaction_digest
JOIN checkpoints c
    ON t1.checkpoint_sequence = c.sequence_number
    AND t2.checkpoint_sequence = c.sequence_number
WHERE
    -- BOTH are write operations
    oc1.change_type IN ('mutated', 'created', 'deleted')
    AND oc2.change_type IN ('mutated', 'created', 'deleted');""")
rows = cur.fetchall()



Conflict graph with only successful transactions

In [ ]:
cur.execute("""SELECT
    oc1.transaction_digest AS tx1,
    oc2.transaction_digest AS tx2,
    oc1.address,
    oc1.input_version AS version,
    c.sequence_number AS checkpoint
FROM object_changes oc1
JOIN object_changes oc2
    ON oc1.address = oc2.address
    AND oc1.transaction_digest < oc2.transaction_digest
    AND oc1.input_version = oc2.input_version
JOIN transactions t1
    ON oc1.transaction_digest = t1.transaction_digest
JOIN transactions t2
    ON oc2.transaction_digest = t2.transaction_digest
JOIN checkpoints c
    ON t1.checkpoint_sequence = c.sequence_number
    AND t2.checkpoint_sequence = c.sequence_number
WHERE
    -- BOTH are write operations
    oc1.change_type IN ('mutated', 'created', 'deleted')
    AND oc2.change_type IN ('mutated', 'created', 'deleted');""")
rows = cur.fetchall()

Compute graph characteristics

In [ ]:
G = nx.Graph()
G.add_edges_from(conflict_edges)

nx.density(G)
nx.diameter(G)
nx.average_degree_connectivity(G)
nx.transitivity(G)
nx.algorithms.coloring.greedy_color(G)
nx.approximation.max_clique(G)
nx.connected_components(G)

Identify smart contract hotspots

In [ ]:
cur.execute("""SELECT address, COUNT(*)
FROM object_changes
WHERE change_type IN ('mutated')
GROUP BY address
ORDER BY COUNT(*) DESC
LIMIT 20;""")


Identify smart contract interaction graphs

In [ ]:
cur.execute("""SELECT kind, COUNT(*)
FROM transactions
GROUP BY kind;""")
